# Fabric field-level lineage demo

Run this notebook **inside a Microsoft Fabric workspace**. It will:

1. Call the [Fabric Scanner APIs](https://learn.microsoft.com/en-us/fabric/governance/metadata-scanning-overview) for every workspace you can see.
2. Parse each semantic model's **M expressions** and **table/column schema** to find which Lakehouse / Warehouse / KQL Database (Eventhouse) column feeds which model column or measure.
3. Print, for **each upstream field**, where it is used downstream (semantic model → table/column/measure).

**Prerequisites**
- The user running this notebook must be a **Fabric Administrator** (Scanner APIs are admin-only).
- Tenant setting *"Enhance admin APIs responses with detailed metadata"* enabled by a Fabric admin (see [Set up metadata scanning](https://learn.microsoft.com/en-us/fabric/admin/metadata-scanning-setup)).

> This is a demo. M-query parsing is regex-based and covers the common shapes (`Lakehouse.Contents`, `Sql.Database`, `AzureDataExplorer.Contents`, `Kusto.Contents`). It won't catch every hand-crafted mashup.

## 1. Config & imports

In [ ]:
import re, time, json, requests
import pandas as pd

# How many workspaces to scan per call (max 100).
BATCH_SIZE = 100
# How far back to look for modified workspaces (days). None = all.
MODIFIED_SINCE_DAYS = None

PBI_BASE = "https://api.powerbi.com/v1.0/myorg/admin"

# In a Fabric notebook this returns a Power BI access token for the current user.
token = notebookutils.credentials.getToken("pbi")
HEADERS = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}


## 2. Run a scan and fetch the metadata

Flow: `GetModifiedWorkspaces` → `PostWorkspaceInfo` (with `lineage`, `datasourceDetails`, `datasetSchema`, `datasetExpressions`) → poll `GetScanStatus` → `GetScanResult`.

In [ ]:
def _get(url):
    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()
    return r.json()

def _post(url, body=None):
    r = requests.post(url, headers=HEADERS, data=json.dumps(body) if body else None)
    r.raise_for_status()
    return r.json() if r.text else {}

# 2a. Modified workspaces
url = f"{PBI_BASE}/workspaces/modified"
if MODIFIED_SINCE_DAYS is not None:
    from datetime import datetime, timedelta, timezone
    since = (datetime.now(timezone.utc) - timedelta(days=MODIFIED_SINCE_DAYS)).strftime("%Y-%m-%dT%H:%M:%S.0000000Z")
    url += f"?modifiedSince={since}"
ws_ids = [w["id"] for w in _get(url)]
print(f"Workspaces to scan: {len(ws_ids)}")

# 2b. Kick off scans in batches
scan_ids = []
post_url = (f"{PBI_BASE}/workspaces/getInfo"
            "?lineage=true&datasourceDetails=true&datasetSchema=true"
            "&datasetExpressions=true&getArtifactUsers=false")
for i in range(0, len(ws_ids), BATCH_SIZE):
    batch = ws_ids[i:i+BATCH_SIZE]
    resp = _post(post_url, {"workspaces": batch})
    scan_ids.append(resp["id"])
print(f"Scans started: {len(scan_ids)}")

# 2c. Poll until done, then fetch results
workspaces = []
for sid in scan_ids:
    while True:
        status = _get(f"{PBI_BASE}/workspaces/scanStatus/{sid}")["status"]
        if status == "Succeeded":
            break
        if status == "Failed":
            raise RuntimeError(f"Scan {sid} failed")
        time.sleep(3)
    result = _get(f"{PBI_BASE}/workspaces/scanResult/{sid}")
    workspaces.extend(result.get("workspaces", []))
print(f"Workspaces returned: {len(workspaces)}")

## 3. Index upstream sources (Lakehouse / Warehouse / Eventhouse)

The Scanner API tells us which Lakehouses/Warehouses/KQL databases each workspace contains. We build a lookup so we can resolve an M-query reference to a concrete artifact.

In [ ]:
from collections import Counter as _Counter

# artifact_id -> {"kind": ..., "name": ..., "workspace": ...}
artifact_by_id = {}
# lowercase name -> list of artifact_ids (fallback lookup)
artifact_by_name = {}
# workspace_id -> list of artifact_ids (same-workspace fallback for Direct Lake)
artifacts_by_workspace = {}
# datasource_instance_id -> connection details (from workspace-level datasourceInstances)
datasource_by_id = {}

# Every variant name the Scanner API uses (across versions) for an upstream item.
# key (lowercased) -> canonical kind
_KEY_TO_KIND = {}
def _add_kind(kind, *aliases):
    for a in aliases:
        _KEY_TO_KIND[a.lower()] = kind

_add_kind("Lakehouse",            "Lakehouse",  "lakehouses",  "Lakehouses")
_add_kind("Warehouse",            "Warehouse",  "warehouses",  "Warehouses")
_add_kind("KQLDatabase",          "KQLDatabase", "kqlDatabases", "KQLDatabases")
_add_kind("KQLDataset",           "KQLDataset", "kqlDatasets",  "KQLDatasets")
_add_kind("SQLDatabase",          "SQLDatabase", "sqlDatabases", "SQLDatabases")
_add_kind("SQLAnalyticsEndpoint", "SQLAnalyticsEndpoint", "sqlAnalyticsEndpoints", "SQLAnalyticsEndpoints")
_add_kind("MirroredDatabase",     "MirroredDatabase", "mirroredDatabases", "MirroredDatabases")
_add_kind("MirroredWarehouse",    "MirroredWarehouse", "mirroredWarehouses", "MirroredWarehouses")
_add_kind("Datamart",             "Datamart", "datamarts", "Datamarts")
_add_kind("Eventhouse",           "Eventhouse", "eventhouses", "Eventhouses")

# Keys we intentionally ignore because they are not upstream data sources.
_IGNORE_KEYS = {
    "reports", "dashboards", "datasets", "dataflows", "notebooks",
    "sparkjobdefinitions", "pipelines", "mlmodels", "mlexperiments",
    "users", "datasourceinstances", "misconfigureddatasourceinstances",
    "artifacts",  # handled separately below
    "reflex", "reflexes", "dataagent", "dataagents", "environments",
}

def _register(wid, wname, aid, aname, kind):
    if not aid:
        return
    artifact_by_id[aid] = {"kind": kind, "name": aname, "workspace": wname, "workspaceId": wid}
    artifact_by_name.setdefault((aname or "").lower(), []).append(aid)
    artifacts_by_workspace.setdefault(wid, []).append(aid)

unknown_keys = _Counter()

for w in workspaces:
    wid, wname = w.get("id"), w.get("name")

    # 1. Walk every top-level key and pick up items in known-type lists.
    for key, items in w.items():
        if not isinstance(items, list) or not items:
            continue
        kind = _KEY_TO_KIND.get(key.lower())
        if kind:
            for a in items:
                if isinstance(a, dict):
                    _register(wid, wname, a.get("id"),
                              a.get("displayName") or a.get("name"), kind)
        elif key.lower() not in _IGNORE_KEYS and isinstance(items[0], dict) and "id" in items[0]:
            unknown_keys[key] += len(items)

    # 2. Some tenants also return a flat "artifacts" list with an artifactType field.
    for a in w.get("artifacts", []) or []:
        if not isinstance(a, dict):
            continue
        raw = a.get("artifactType") or a.get("type") or ""
        kind = _KEY_TO_KIND.get(raw.lower())
        if kind:
            _register(wid, wname, a.get("id"),
                      a.get("displayName") or a.get("name"), kind)
        elif raw and raw.lower() not in _IGNORE_KEYS:
            unknown_keys[f"artifacts[{raw}]"] += 1

    # 3. Datasource instances live at workspace scope in the scan.
    for di in w.get("datasourceInstances", []) or []:
        did = di.get("datasourceId") or di.get("id")
        if did:
            datasource_by_id[did] = di

print(f"Upstream artifacts indexed: {len(artifact_by_id)}")
kinds = _Counter(info["kind"] for info in artifact_by_id.values())
for k, v in sorted(kinds.items()):
    print(f"  {k}: {v}")
print(f"Datasource instances indexed: {len(datasource_by_id)}")

if unknown_keys:
    print("\n[hint] Scan returned these unrecognised list-valued keys (may be new Fabric item types):")
    for k, v in unknown_keys.most_common(20):
        print(f"  {k}: {v} item(s)")
    print("  If any of these look like an upstream data source, add them to _add_kind above.")


## 4. Parse semantic-model M expressions → upstream (artifact, schema, table)

For each table in each semantic model we look at its `source` expression (Power Query M) and pull out the artifact, schema and table it points to.

In [ ]:
# Patterns for the common Fabric connectors.
RE_LAKEHOUSE = re.compile(
    r'Lakehouse\.Contents\([^)]*\)(?P<rest>.*?)(?=(?:in\s|\Z))',
    re.S | re.I,
)
RE_SQL       = re.compile(r'Sql\.Databases?\(\s*"(?P<srv>[^"]+)"[^)]*\)', re.I)
RE_KQL       = re.compile(r'(?:AzureDataExplorer|Kusto)\.Contents\(\s*"(?P<srv>[^"]+)"', re.I)

# Navigation step: {[Workspace="..",Id="..",ItemKind="Lakehouse",Item="..",Schema=".."]}
RE_NAV       = re.compile(r'\{\s*\[(?P<props>[^\]]+)\]\s*\}', re.S)
RE_PROP      = re.compile(r'(\w+)\s*=\s*"([^"]*)"')

def _nav_props(expr: str):
    """Return list of dicts, one per {[...]} navigation step in M source."""
    return [dict(RE_PROP.findall(m.group("props"))) for m in RE_NAV.finditer(expr or "")]

def resolve_upstream(expr: str):
    """Return (kind, artifact_name, workspace_name, schema, table) or None."""
    if not expr:
        return None
    props_list = _nav_props(expr)
    # Merge across navigation steps – later steps override earlier ones.
    merged = {}
    for p in props_list:
        merged.update(p)

    schema = merged.get("Schema")
    table  = merged.get("Item") or merged.get("Name")
    kind   = merged.get("ItemKind")
    art_id = merged.get("Id") or merged.get("ItemId")
    art_nm = merged.get("LakehouseName") or merged.get("WarehouseName") or merged.get("DatabaseName")

    artifact, workspace = None, None
    if art_id and art_id in artifact_by_id:
        info = artifact_by_id[art_id]
        artifact, workspace, kind = info["name"], info["workspace"], info["kind"]
    elif art_nm:
        hits = artifact_by_name.get(art_nm.lower(), [])
        if len(hits) == 1:
            info = artifact_by_id[hits[0]]
            artifact, workspace, kind = info["name"], info["workspace"], info["kind"]
        else:
            artifact = art_nm

    if not kind:
        if RE_LAKEHOUSE.search(expr): kind = "Lakehouse"
        elif RE_KQL.search(expr):     kind = "KQLDatabase"
        elif RE_SQL.search(expr):     kind = "Warehouse/SQL"

    if not (kind or artifact or table):
        return None
    return (kind or "Unknown", artifact or "?", workspace or "?", schema or "?", table or "?")

## 5. Build the field-level lineage table

For each semantic-model table we assume its columns map by **name** to columns in the upstream table (renames in M are not yet resolved — that's a good next step). Measures are attributed to every column referenced in their DAX text.

In [ ]:
RE_DAX_COLREF = re.compile(r"'?(?P<tbl>[A-Za-z0-9_ ]+?)'?\[(?P<col>[^\]]+)\]")
RE_GUID       = re.compile(r"\b[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}\b")

LINEAGE_COLUMNS = [
    "upstream", "upstream_kind", "upstream_item", "upstream_workspace",
    "upstream_field", "downstream_kind", "downstream_item", "downstream_workspace",
    "resolution",  # how we resolved the upstream
]

# Preferred order when the workspace has several upstream artifacts.
_KIND_PRIORITY = {"Lakehouse": 0, "Warehouse": 1, "MirroredDatabase": 2,
                  "KQLDatabase": 3, "SQLAnalyticsEndpoint": 4, "SQLDatabase": 5,
                  "Datamart": 6}


def _collect_expression(table_obj) -> str:
    """Return the concatenated M expression for a semantic-model table."""
    chunks = []
    src = table_obj.get("source")
    if isinstance(src, list):
        for s in src:
            if isinstance(s, dict) and s.get("expression"):
                chunks.append(s["expression"])
            elif isinstance(s, str):
                chunks.append(s)
    elif isinstance(src, dict) and src.get("expression"):
        chunks.append(src["expression"])
    elif isinstance(src, str):
        chunks.append(src)

    for p in table_obj.get("partitions", []) or []:
        psrc = (p or {}).get("source") or {}
        if isinstance(psrc, dict) and psrc.get("expression"):
            chunks.append(psrc["expression"])
        elif isinstance(psrc, str):
            chunks.append(psrc)

    if table_obj.get("expression"):
        chunks.append(table_obj["expression"])

    return "\n".join(chunks)


def _table_entity_schema(table_obj):
    """Return (entity_name, schema_name) from a Direct Lake table.source, if present."""
    entity, schema = None, None
    src = table_obj.get("source")
    if isinstance(src, list):
        for s in src:
            if isinstance(s, dict):
                entity = entity or s.get("entityName") or s.get("Item")
                schema = schema or s.get("schemaName") or s.get("Schema")
    elif isinstance(src, dict):
        entity = src.get("entityName") or src.get("Item")
        schema = src.get("schemaName") or src.get("Schema")
    return entity, schema


def _dataset_upstream_ids(ds):
    """Return artifact IDs the dataset lists as upstream (direct link)."""
    ids = []
    for key in ("upstreamArtifacts", "upstreamDatasources", "upstreamDatasets",
                "upstreamDataflows"):
        for u in ds.get(key, []) or []:
            if isinstance(u, dict):
                aid = u.get("artifactId") or u.get("id") or u.get("targetId")
                if aid:
                    ids.append(aid)
            elif isinstance(u, str):
                ids.append(u)
    return ids


def _dataset_datasource_ids(ds):
    """Return datasourceInstance IDs the dataset uses."""
    ids = []
    for u in ds.get("datasourceUsages", []) or []:
        if isinstance(u, dict):
            did = u.get("datasourceInstanceId") or u.get("id")
            if did:
                ids.append(did)
    return ids


def _guids_in_dataset(ds):
    """Recursively yield every GUID that appears anywhere in the dataset dict."""
    stack = [ds]
    seen = set()
    while stack:
        v = stack.pop()
        if isinstance(v, dict):
            stack.extend(v.values())
        elif isinstance(v, list):
            stack.extend(v)
        elif isinstance(v, str):
            for g in RE_GUID.findall(v):
                if g not in seen:
                    seen.add(g)
                    yield g


def _pick_same_workspace(ids):
    """From a list of artifact IDs, pick the highest-priority upstream artifact."""
    infos = [artifact_by_id[a] for a in ids if a in artifact_by_id]
    if not infos:
        return None
    infos.sort(key=lambda i: _KIND_PRIORITY.get(i["kind"], 99))
    return infos[0]


def resolve_table_upstream(table_obj, ds, ds_workspace_id):
    """Try each resolver in turn. Returns (kind, art, ws, schema, up_tbl, how) or None."""
    entity, schema = _table_entity_schema(table_obj)
    tname = table_obj.get("name")

    up = resolve_upstream(_collect_expression(table_obj))
    if up:
        return (*up, "M-parse")

    for aid in _dataset_upstream_ids(ds):
        if aid in artifact_by_id:
            info = artifact_by_id[aid]
            return (info["kind"], info["name"], info["workspace"],
                    schema or "?", entity or tname or "?", "upstreamArtifacts")

    for did in _dataset_datasource_ids(ds):
        di = datasource_by_id.get(did) or {}
        details = di.get("connectionDetails") or {}
        srv = (details.get("server") or details.get("url") or "").lower()
        for aid, info in artifact_by_id.items():
            if info["name"] and info["name"].lower() in srv:
                return (info["kind"], info["name"], info["workspace"],
                        schema or "?", entity or tname or "?", "datasourceUsage")

    ids_in_ds = [g for g in _guids_in_dataset(ds) if g in artifact_by_id]
    info = _pick_same_workspace(ids_in_ds)
    if info:
        return (info["kind"], info["name"], info["workspace"],
                schema or "?", entity or tname or "?", "guid-in-dataset")

    info = _pick_same_workspace(artifacts_by_workspace.get(ds_workspace_id, []))
    if info:
        return (info["kind"], info["name"], info["workspace"],
                schema or "?", entity or tname or "?", "same-workspace guess")

    return None


rows = []
unresolved_tables = []

for w in workspaces:
    for ds in w.get("datasets", []) or []:
        ds_name = ds.get("name")
        ds_ws   = w.get("name")
        ds_wid  = w.get("id")
        tables  = ds.get("tables", []) or []

        table_upstream = {}
        table_cols     = {}
        for t in tables:
            tname = t.get("name")
            table_cols[tname] = {c.get("name") for c in (t.get("columns") or [])}
            table_upstream[tname] = resolve_table_upstream(t, ds, ds_wid)
            if table_upstream[tname] is None:
                unresolved_tables.append((ds_ws, ds_name, tname))

        for tname, up in table_upstream.items():
            if not up:
                continue
            kind, art, ws, schema, up_tbl, how = up
            for col in table_cols.get(tname, []):
                rows.append({
                    "upstream":       f"{kind}:{art}.{schema}.{up_tbl}.{col}",
                    "upstream_kind":  kind,
                    "upstream_item":  art,
                    "upstream_workspace": ws,
                    "upstream_field": f"{schema}.{up_tbl}.{col}",
                    "downstream_kind":    "SemanticModel.Column",
                    "downstream_item":    f"{ds_name}.{tname}.{col}",
                    "downstream_workspace": ds_ws,
                    "resolution": how,
                })

        for t in tables:
            for m in t.get("measures", []) or []:
                mname, mexpr = m.get("name"), m.get("expression") or ""
                refs = {(rt.strip(), rc.strip()) for rt, rc in RE_DAX_COLREF.findall(mexpr)}
                for rt, rc in refs:
                    up = table_upstream.get(rt)
                    if not up or rc not in table_cols.get(rt, set()):
                        continue
                    kind, art, ws, schema, up_tbl, how = up
                    rows.append({
                        "upstream":       f"{kind}:{art}.{schema}.{up_tbl}.{rc}",
                        "upstream_kind":  kind,
                        "upstream_item":  art,
                        "upstream_workspace": ws,
                        "upstream_field": f"{schema}.{up_tbl}.{rc}",
                        "downstream_kind":    "SemanticModel.Measure",
                        "downstream_item":    f"{ds_name}.[{mname}]",
                        "downstream_workspace": ds_ws,
                        "resolution": how,
                    })

lineage = pd.DataFrame(rows, columns=LINEAGE_COLUMNS)
if not lineage.empty:
    lineage = lineage.drop_duplicates().sort_values(["upstream", "downstream_item"])


# ---------- Reusable filter + lookup, used here and in cell 9 ----------

def apply_filters(df, item=None, schema=None, table=None, fields=None):
    """Filter a lineage-shaped DataFrame by upstream_item / schema / table / field(s).
    All matches are case-insensitive substring matches. `fields` may be a
    string or a list of strings (OR-combined)."""
    out = df
    if item:
        out = out[out["upstream_item"].str.contains(item, case=False, na=False, regex=False)]
    parts = out["upstream_field"].str.split(".", n=2, expand=True) if len(out) else None
    if schema:
        out = out[parts[0].fillna("").str.contains(schema, case=False, na=False, regex=False)]
        parts = out["upstream_field"].str.split(".", n=2, expand=True) if len(out) else None
    if table:
        out = out[parts[1].fillna("").str.contains(table, case=False, na=False, regex=False)]
        parts = out["upstream_field"].str.split(".", n=2, expand=True) if len(out) else None
    if fields:
        fl = [fields] if isinstance(fields, str) else list(fields)
        col = parts[2].fillna("") if parts is not None else pd.Series([], dtype=str)
        mask = pd.Series(False, index=out.index)
        for f in fl:
            mask = mask | col.str.contains(f, case=False, na=False, regex=False)
        out = out[mask]
    return out


def lookup(item=None, schema=None, table=None, fields=None) -> pd.DataFrame:
    """Return downstream usages of upstream fields matching the given filters."""
    hits = apply_filters(lineage, item=item, schema=schema, table=table, fields=fields)
    if hits.empty:
        print(f"No downstream usage for item={item!r}, schema={schema!r}, "
              f"table={table!r}, fields={fields!r}.")
        return hits
    print(f"Matched {hits['upstream_field'].nunique()} upstream field(s), "
          f"used by {hits['downstream_item'].nunique()} downstream object(s).")
    return hits[["upstream", "downstream_kind", "downstream_item", "downstream_workspace"]]


print(f"Lineage edges: {len(lineage)}")
if not lineage.empty:
    print("Edges by resolution method:")
    print(lineage["resolution"].value_counts().to_string())
else:
    print("No lineage edges found. Run the diagnostic cell below to see why.")
lineage.head(20)


## 5b. Diagnostic — what did the scan actually return?

Run this cell if "Lineage edges: 0". It prints per-workspace counts and shows sample M expressions so you can tell whether the scan is empty, semantic models are missing, or the parser just doesn't match your M shape.


In [ ]:
from collections import Counter

n_ws              = len(workspaces)
n_datasets        = 0
n_ds_with_tables  = 0
n_ds_with_upstream= 0
n_tables          = 0
n_with_expr       = 0
resolutions       = Counter()
samples           = []
unresolved        = []

for w in workspaces:
    wid = w.get("id")
    for ds in w.get("datasets", []) or []:
        n_datasets += 1
        tables = ds.get("tables", []) or []
        if tables:
            n_ds_with_tables += 1
        ups = _dataset_upstream_ids(ds)
        dus = _dataset_datasource_ids(ds)
        if any(a in artifact_by_id for a in ups):
            n_ds_with_upstream += 1
        for t in tables:
            n_tables += 1
            expr = _collect_expression(t)
            if expr.strip():
                n_with_expr += 1
            result = resolve_table_upstream(t, ds, wid)
            if result is None:
                resolutions["unresolved"] += 1
                unresolved.append((w.get("name"), ds.get("name"), t.get("name"), expr, ups, dus))
            else:
                resolutions[result[5]] += 1  # the "how" tag
            if len(samples) < 3:
                samples.append((w.get("name"), ds.get("name"), t.get("name"), expr, ups, dus))

print(f"Workspaces returned by scan     : {n_ws}")
print(f"Semantic models (datasets)      : {n_datasets}")
print(f"  ...with any tables returned   : {n_ds_with_tables}")
print(f"  ...with upstream artifact link: {n_ds_with_upstream}")
print(f"Tables in those models          : {n_tables}")
print(f"Tables with any expression      : {n_with_expr}")
print("Tables by resolution method     :")
for k, v in resolutions.most_common():
    print(f"  {k}: {v}")

print("\nUpstream-artifact index (Lakehouse/Warehouse/KQL/…):")
kinds = Counter(info["kind"] for info in artifact_by_id.values())
for k, v in sorted(kinds.items()):
    print(f"  {k}: {v}")
print(f"Datasource instances indexed    : {len(datasource_by_id)}")

if n_datasets and n_ds_with_tables < n_datasets:
    missing = n_datasets - n_ds_with_tables
    print(f"\n[WARN] {missing}/{n_datasets} semantic models returned NO tables/columns.")
    print("       Tenant admin setting 'Enhance admin APIs responses with detailed metadata'")
    print("       must be ON and applied to the security group owning those workspaces.")

if samples:
    print("\n--- Sample tables (first 3) ---")
    for wname, dname, tname, expr, ups, dus in samples:
        print(f"\n[{wname}] {dname} / {tname}")
        print(f"  upstreamArtifacts : {ups or '(none)'}")
        print(f"  datasourceUsages  : {dus or '(none)'}")
        print(f"  expression        : {expr[:400]!r}{'...' if len(expr) > 400 else ''}")

if unresolved:
    print("\n--- First unresolved table ---")
    wname, dname, tname, expr, ups, dus = unresolved[0]
    print(f"[{wname}] {dname} / {tname}")
    print(f"  upstreamArtifacts : {ups}")
    print(f"  datasourceUsages  : {dus}")
    print(f"  expression        : {expr[:1500]}")
    print("\nIf both link lists are empty and the M expression is trivial (just the table name),")
    print("this dataset is a Direct Lake model whose upstream Lakehouse/Warehouse was not in")
    print("the scan. Either it lives in a workspace your admin token cannot see, or the workspace")
    print("was not returned by GetModifiedWorkspaces (set MODIFIED_SINCE_DAYS=None and re-run).")


In [ ]:
# --- Find a missing item across the scan result ---
SEARCH = "kaggle"   # substring to look for (case-insensitive)

print(f"All {len(workspaces)} workspaces returned by the scan:")
for w in sorted(workspaces, key=lambda x: (x.get("name") or "").lower()):
    n_ds = len(w.get("datasets", []) or [])
    n_lh = len(w.get("lakehouses", []) or [])
    n_wh = len(w.get("warehouses", []) or [])
    n_kq = len(w.get("kqlDatabases", []) or [])
    n_rp = len(w.get("reports", []) or [])
    print(f"  {w.get('name'):40s}  datasets={n_ds:2d}  lh={n_lh:2d}  wh={n_wh:2d}  kql={n_kq:2d}  reports={n_rp:2d}")

print(f"\nSearching every artifact key for '{SEARCH}':")
hits = []
for w in workspaces:
    for key, items in w.items():
        if not isinstance(items, list):
            continue
        for a in items:
            if not isinstance(a, dict):
                continue
            name = a.get("name") or a.get("displayName") or ""
            if SEARCH.lower() in name.lower():
                hits.append((w.get("name"), key, name, a.get("id")))
if hits:
    for ws, key, name, aid in hits:
        print(f"  [{ws}] {key} -> {name}  ({aid})")
else:
    print("  (no matches – this item is not in the scan)")
    print("\nLikely causes:")
    print(" - The workspace containing it is a personal workspace (My workspace).")
    print(" - GetModifiedWorkspaces returned inactive-workspace filter kicked in.")
    print("   Try re-running cell 2 with explicit params (see suggested edit below).")


## 6. Query the lineage — "if I change X, what breaks?"

`lookup(item=..., schema=..., table=..., fields=...)` (defined in cell 5) filters the `lineage` DataFrame. Each argument is an optional case-insensitive substring match; `fields` accepts a single name or a list. Omit an argument to skip that filter.


In [ ]:
# Change the call below to explore. Only the LAST expression in a cell is shown.

lookup(item="Kaggle", table="betfront", fields="COUNTRY")

# --- Alternative views (comment out the call above and uncomment ONE) ---
#
# lookup(item="Kaggle")                       # everything fed by the Kaggle Lakehouse
# apply_filters(lineage)                      # the whole raw lineage table
#
# (apply_filters(lineage)                     # grouped "where is each field used"
#     .groupby(["upstream_kind", "upstream_workspace", "upstream_item", "upstream_field"])
#     .agg(downstream_count=("downstream_item", "nunique"),
#          downstream_items=("downstream_item", lambda s: sorted(set(s))))
#     .reset_index()
#     .sort_values("downstream_count", ascending=False))


## 7. Which report field is used in which visual?

The Scanner API doesn't tell you which visual in a Power BI report uses which model field. For that we call the **Fabric REST API** `Reports - Get Definition` (PBIR format), which returns the report as a set of JSON parts. Every column or measure a visual binds to appears in those JSON parts as `[TableName].[ColumnName]`, so a simple regex extracts them.

We keep the mapping at report level (visual/page parsing is possible but noisier); the result is `report → semantic model → table → field`.


In [ ]:
import base64

# Fabric REST needs a token with audience https://api.fabric.microsoft.com.
_fabric_token = None
for aud in ("fabric", "https://api.fabric.microsoft.com", "https://api.fabric.microsoft.com/.default"):
    try:
        _fabric_token = notebookutils.credentials.getToken(aud)
        if _fabric_token:
            print(f"Got Fabric token via audience: {aud!r}")
            break
    except Exception as e:
        print(f"  getToken({aud!r}) failed: {e}")
if not _fabric_token:
    _fabric_token = token
    print("Falling back to the pbi token for Fabric REST calls.")
FABRIC_HEADERS = {"Authorization": f"Bearer {_fabric_token}", "Content-Type": "application/json"}


dataset_by_id = {}
for w in workspaces:
    for ds in w.get("datasets", []) or []:
        if ds.get("id"):
            dataset_by_id[ds["id"]] = {"name": ds.get("name"), "workspace": w.get("name")}

FABRIC = "https://api.fabric.microsoft.com/v1"

# Patterns that occur in PBIR JSON payloads for column / measure references.
RE_BRACKETED = re.compile(r'\[([^\[\]]+)\]\.\[([^\[\]]+)\]')                 # [Table].[Column]
RE_NAME_KEY  = re.compile(r'"Name"\s*:\s*"([^"\.]+)\.([^"]+)"')              # "Name":"Table.Column"
RE_PROPERTY  = re.compile(r'"Property"\s*:\s*"([^"]+)"')                     # column name in Select clause
RE_ENTITY    = re.compile(r'"Entity"\s*:\s*"([^"]+)"')                       # table name in From clause


def _extract_field_refs(text: str):
    """Return set of (table, column) tuples from a PBIR JSON string."""
    hits = set()
    hits.update(RE_BRACKETED.findall(text))
    hits.update(RE_NAME_KEY.findall(text))
    # Combine every Entity with every Property in the same visual JSON as a
    # coarse fallback -- this over-generates a bit but never misses.
    entities   = RE_ENTITY.findall(text)
    properties = RE_PROPERTY.findall(text)
    for e in entities:
        for p in properties:
            hits.add((e, p))
    return hits


def _get_report_definition(ws_id, report_id, poll_max=20, poll_delay=3):
    url = f"{FABRIC}/workspaces/{ws_id}/reports/{report_id}/getDefinition?format=PBIR"
    r = requests.post(url, headers=FABRIC_HEADERS)
    if r.status_code == 200:
        return r.json()
    if r.status_code != 202:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
    op_id = r.headers.get("x-ms-operation-id")
    if not op_id:
        raise RuntimeError("202 without x-ms-operation-id")
    for _ in range(poll_max):
        time.sleep(poll_delay)
        s = requests.get(f"{FABRIC}/operations/{op_id}", headers=FABRIC_HEADERS)
        if s.status_code != 200:
            raise RuntimeError(f"poll HTTP {s.status_code}")
        js = s.json()
        status = js.get("status")
        if status == "Succeeded":
            res = requests.get(f"{FABRIC}/operations/{op_id}/result", headers=FABRIC_HEADERS)
            if res.status_code != 200:
                raise RuntimeError(f"result HTTP {res.status_code}: {res.text[:200]}")
            return res.json()
        if status == "Failed":
            err = js.get("error") or {}
            raise RuntimeError(f"operation failed: {err.get('errorCode')} {err.get('message')}")
    raise RuntimeError("polling timed out")


report_rows = []
report_errors = []
per_report_counts = []

for w in workspaces:
    for rp in w.get("reports", []) or []:
        ds_id = rp.get("datasetId")
        ds_info = dataset_by_id.get(ds_id) if ds_id else None
        if not ds_info:
            continue
        try:
            defn = _get_report_definition(w["id"], rp["id"])
        except Exception as e:
            report_errors.append((w.get("name"), rp.get("name"), str(e)))
            continue
        if not defn:
            report_errors.append((w.get("name"), rp.get("name"), "empty definition"))
            continue

        found_in_report = set()
        for part in defn.get("definition", {}).get("parts", []) or []:
            payload = part.get("payload")
            if not payload:
                continue
            try:
                text = base64.b64decode(payload).decode("utf-8", errors="ignore")
            except Exception:
                continue
            for tbl, col in _extract_field_refs(text):
                found_in_report.add((tbl, col))

        per_report_counts.append((w.get("name"), rp.get("name"), len(found_in_report)))
        for tbl, col in found_in_report:
            report_rows.append({
                "report_workspace":         w.get("name"),
                "report_name":              rp.get("name"),
                "report_id":                rp.get("id"),
                "semantic_model":           ds_info["name"],
                "semantic_model_workspace": ds_info["workspace"],
                "table":                    tbl,
                "field":                    col,
            })

report_usage = pd.DataFrame(
    report_rows,
    columns=["report_workspace", "report_name", "report_id",
             "semantic_model", "semantic_model_workspace", "table", "field"],
).drop_duplicates()

print(f"\nReport-field usages extracted: {len(report_usage)}")
if per_report_counts:
    print("Fields extracted per report:")
    for ws, rp, n in per_report_counts:
        print(f"  [{ws}] {rp}: {n}")
if report_errors:
    print(f"\nReports skipped ({len(report_errors)}):")
    for ws, rp, msg in report_errors[:10]:
        print(f"  [{ws}] {rp}: {msg}")
report_usage.head(20)


## 8. End-to-end: Lakehouse/Warehouse field → Semantic model column → Report

Join `lineage` (upstream data source → semantic-model column) with `report_usage` (report → semantic-model column) on `(model, table, field)` to get the full chain. Use `full_lineage(item=..., schema=..., table=..., fields=...)` to answer "if I change this column in the Lakehouse, which report(s) break?" — same filter arguments as `lookup()`.


In [ ]:
col_edges = lineage[lineage["downstream_kind"] == "SemanticModel.Column"].copy()
if not col_edges.empty:
    parts = col_edges["downstream_item"].str.split(".", n=2, expand=True)
    col_edges["model"] = parts[0]
    col_edges["table"] = parts[1]
    col_edges["field"] = parts[2]

if col_edges.empty:
    print("`lineage` has no SemanticModel.Column edges. Re-run cell 5.")
    full = pd.DataFrame()
elif report_usage.empty:
    print("`report_usage` is empty. Re-run cell 7 (check its output for extraction counts).")
    full = pd.DataFrame()
else:
    full = col_edges.merge(
        report_usage,
        left_on=["model", "table", "field"],
        right_on=["semantic_model", "table", "field"],
        how="inner",
    )[[
        "upstream_kind", "upstream_workspace", "upstream_item", "upstream_field",
        "model", "semantic_model_workspace",
        "report_workspace", "report_name",
        "resolution",
    ]].drop_duplicates().rename(columns={"model": "semantic_model"})
    print(f"End-to-end edges: {len(full)}")


def full_lineage(item=None, schema=None, table=None, fields=None) -> pd.DataFrame:
    """Same filter args as lookup(), but returns Lakehouse -> Model -> Report rows."""
    return apply_filters(full, item=item, schema=schema, table=table, fields=fields)


# Change the call below to explore. Only the LAST expression in a cell is shown.

full_lineage(item="Kaggle", fields="COUNTRY")

# --- Alternative views ---
# full_lineage(item="Kaggle")   # every Kaggle field that ends up in a report
# full                          # the whole end-to-end table
